#Drive Mounting

#Variable Loading

In [20]:
%pip install pandas


[notice] A new release of pip is available: 25.0 -> 25.2
[notice] To update, run: python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [21]:

file_path = "/workspace/Variables/Copy of shared_variable.pkl"

import pickle

with open(file_path, 'rb') as file:
    loaded_variable = pickle.load(file)
print(loaded_variable)

0     What work, tradition or theory does Spaceballs...
1     What work, tradition or theory does Bullet Tra...
2     What work, tradition or theory does Q Who refe...
3     What work, tradition or theory does The Tin Dr...
4     What work, tradition or theory does Back to th...
5     What is the dominant foot or preferred stance ...
6     What is the dominant foot or preferred stance ...
7     What is the dominant foot or preferred stance ...
8     What is the dominant foot or preferred stance ...
9     Where was Ahwak recorded, Olympic Studios or A...
10    Where was The Snow Queen recorded, Õru or Tall...
11    Where was The Lost Treasure for Aquila recorde...
12    Where was Hinatazaka de Aimashō recorded, Tele...
13    Where was Soundtrack recorded, Fullerton Colle...
14    What is the tempo marking for Beauty and the B...
15    What is the tempo marking for We No Speak Amer...
16    What is the tempo marking for Baa, Baa, Black ...
17    Under what copyright license was MuLinux r

In [22]:
import pandas as pd

translated_data = pd.DataFrame(loaded_variable)


translated_data.head()

,SAE output
0,"What work, tradition or theory does Spaceballs..."
1,"What work, tradition or theory does Bullet Tra..."
2,"What work, tradition or theory does Q Who refe..."
3,"What work, tradition or theory does The Tin Dr..."
4,"What work, tradition or theory does Back to th..."


In [23]:
!nvidia-smi

Sat Oct 18 01:56:44 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        On  |   00000000:82:00.0 Off |                  Off |
|  0%   30C    P8             13W /  450W |       1MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

#Retrieving Entities + Relations with FALCON

In [24]:
%pip install requests


[notice] A new release of pip is available: 25.0 -> 25.2
[notice] To update, run: python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [25]:
import requests


#Getting entities for datasets

In [26]:
# url= "https://labs.tib.eu/falcon/falcon2/api?mode=long"

# def get_entities(translated_data):
#   answers = []
#   for i in range(len(translated_data)):
#     question = translated_data.iloc[i,0]
#     data = {f'text': question}
#     headers = {"Content-Type": "application/json"}
#     answer = requests.post(url,json=data, headers=headers)
#     answers.append(answer.json())
#   return answers







#Getting entities for single questions

In [27]:
url= "https://labs.tib.eu/falcon/falcon2/api?mode=long"

def get_entities(question):
    data = {f'text': question}
    headers = {"Content-Type": "application/json"}
    answer = requests.post(url,json=data, headers=headers)
    return answer.json()


In [28]:
question = ("What is the first book published in the Lord of the Rings book trilogy?")

In [29]:
answers = get_entities(question)

In [30]:
print(answers)

{'entities_wikidata': [{'URI': 'http://www.wikidata.org/entity/Q190214', 'surface form': 'Lord of the Rings book trilogy'}, {'URI': 'http://www.wikidata.org/entity/Q56431177', 'surface form': 'book'}], 'relations_wikidata': [{'URI': 'http://www.wikidata.org/entity/P577', 'surface form': 'published'}]}


In [31]:
def extract_entities(answers):
    entity_ids = []
    relation_ids = []
    # Entity extraction
    for e in answers.get('entities_wikidata', []):
        uri = e.get('URI', '')
        if uri.startswith('http://www.wikidata.org/entity/'):
            entity_ids.append(uri.split('/')[-1]) #Getting the entity name and storing it in a list

    # Extract relations
    for r in answers.get('relations_wikidata', []):
        uri = r.get('URI', '')
        if uri.startswith('http://www.wikidata.org/entity/'):
            relation_ids.append(uri.split('/')[-1])

    return {
        'entities': entity_ids,
        'relations': relation_ids
    }

results = extract_entities(answers)

print(results)

{'entities': ['Q190214', 'Q56431177'], 'relations': ['P577']}


In [32]:
str(results)

"{'entities': ['Q190214', 'Q56431177'], 'relations': ['P577']}"

In [33]:
def entity_as_l(results):
 entities = results.get("entities")
 return entities

entity = entity_as_l(results)
print(entity)

['Q190214', 'Q56431177']


In [34]:
def relation_as_l(results):
  relation = (results.get("relations"))
  return relation

relation = relation_as_l(results)

#Text to SPARQl with QWEN

In [35]:
# !pip install bitsandbytes accelerate

In [36]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 90.7 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 25.0 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip


In [37]:
%pip install transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 31.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.6/806.6 kB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 kB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 75.1 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.0 -> 25.2
[notice] To update, run: python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [38]:
%pip install transformers


[notice] A new release of pip is available: 25.0 -> 25.2
[notice] To update, run: python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [39]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


save_path = "/workspace/Algoverse_KSAC/Models/Qwen3-8B"

evaltok = AutoTokenizer.from_pretrained(save_path, local_files_only=True, enable_thinking = False)

# bNb_config = BitsAndBytesConfig( #4-bit quant
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype="float16",
#     bnb_4bit_use_double_quant=True,

# )


evalmodel = AutoModelForCausalLM.from_pretrained(
    save_path,
    local_files_only=True,
    dtype='auto',
    device_map="auto",
    # quantization_config = bNb_config
)

print("Reloaded model successfully")
print(f"model.device = {evalmodel.device}") #Verifying device

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 5/5 [00:44<00:00,  8.87s/it]

Reloaded model successfully
model.device = cuda:0


In [40]:
!nvidia-smi

Sat Oct 18 01:57:48 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        On  |   00000000:82:00.0 Off |                  Off |
|  0%   33C    P2             45W /  450W |   16038MiB /  24564MiB |      3%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [41]:
def evaluate(question, entity, relation):
    messages = [
        {
            "role": "system",
            "content": (
   """ You are a SPARQL query constructor for Wikidata.

Your task is to generate a syntactically correct and executable SPARQL query given the provided question, entities (QIDs), and relations (PIDs).

Rules:
1. Output only a single SPARQL query string. Treat it as one continuous string, not a list or array.
2. Do NOT include any explanations, comments, markdown formatting, or metadata.
3. Do NOT add label services (e.g., SERVICE wikibase:label) or prefixes.
4. Do NOT use commas or string formatting symbols (such as quotes or brackets) inside the output.
5. Do NOT wrap your output in parentheses or quotation marks.
6. The output must always begin in the form:
   SELECT ?variable WHERE { ... } where variable is something contextually relevant to the question
7. Generate only the minimal triple patterns necessary for the given entities and relations.
8. add the wd: clause in front of queries and the wdt: clause in front of relations
9. The direction of the properties must be semantically correct.
10. For the property P179 ("part of the series")
    - If the given entity represents a specific work reverse the direction so that the query retrieves all items that share the same series.

Your output must be executable as-is on the Wikidata SPARQL endpoint."""




            )
        },
        {
            "role": "user",
            "content": f"question: {question}, entity: {entity}, relation: {relation}"
        }
    ]

    inputs = evaltok.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking = False
    ).to(evalmodel.device)

    outputs = evalmodel.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        temperature=0.1,
        eos_token_id=evaltok.eos_token_id,

    )

    output_answer = evaltok.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],  # Slice off input tokens to get only the generated continuation
        skip_special_tokens=True
    ).split('\n')

    return output_answer


query = evaluate(question,entity, relation)
print(query)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


['SELECT ?book WHERE { wd:Q190214 wdt:P577 ?publicationDate . ?book wdt:P577 ?publicationDate . ?book wdt:P179 wd:Q190214 . }']


In [42]:
def clean(query):
    cleaned = [line.replace(",", "") for line in query]  # remove commas
    return " ".join(cleaned).strip()  # join into one string and trim spaces


query = clean(query).strip()


In [43]:
print(query)

SELECT ?book WHERE { wd:Q190214 wdt:P577 ?publicationDate . ?book wdt:P577 ?publicationDate . ?book wdt:P179 wd:Q190214 . }


#Retrieval from the wikidata API

In [44]:

USER_AGENT = "Colab-SAETOWikidata-SPARQL/1.0 (contact: saketsan8@gmail.com)"
WD_SEARCH_API = "https://www.wikidata.org/w/api.php"
WDQS_ENDPOINT = "https://query.wikidata.org/sparql"
import json





def run_sparql(query, user_agent=USER_AGENT):
    headers = {
        "Accept": "application/sparql-results+json",
        "User-Agent": user_agent
    }
    r = requests.get(WDQS_ENDPOINT, params={"query": query}, headers=headers, timeout=60)
    r.raise_for_status()
    data = r.json()
    simplified = []
    # Loop through each result in "bindings"
    for binding in data.get("results", {}).get("bindings", []):
        clean_entry = {}
        for key, val in binding.items():
          value = val.get("value", "")
          clean_entry[key] = value

        if clean_entry:
            simplified.append(clean_entry)
    return simplified
# Example SPARQL query — find Einstein's wives



QUERY = query
# Run and print the clean structured result
result = run_sparql(QUERY)

context = json.dumps(result, indent=2, ensure_ascii=False)

print(context)



[
  {
    "book": "http://www.wikidata.org/entity/Q127367"
  }
]


In [45]:
links = []
for d in result:
  for v in d.values():
    links.append(v)

In [46]:


def get_name(link, user_agent=USER_AGENT):
    entity_id = link.split("/")[-1].strip()
    url = WD_SEARCH_API
    params = {
        "action": "wbgetentities",
        "ids": entity_id,
        "format": "json",
        "languages": "en"
    }
    headers = {"User-Agent": user_agent}
    r = requests.get(url, params=params, headers=headers)
    data = r.json()
    return data["entities"][entity_id]["labels"]["en"]["value"]
for l in links:
  print(get_name(l))



The Lord of the Rings: The Fellowship of the Ring


In [47]:
result

result_named = [{k:get_name(v)} for d in result for k,v in d.items()]

In [48]:
result_named

[{'book': 'The Lord of the Rings: The Fellowship of the Ring'}]

LLM QA

In [63]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

save_path = "/workspace/Algoverse_KSAC/Models/flan-base"

tokenizer = AutoTokenizer.from_pretrained(save_path, local_files_only=True, enable_thinking = False)

# bNb_config = BitsAndBytesConfig( #4-bit quant
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype="float16",
#     bnb_4bit_use_double_quant=True,

# )


model = AutoModelForSeq2SeqLM.from_pretrained(
    save_path,
    local_files_only=True,
    dtype='auto',
    device_map="auto",
    # quantization_config = bNb_config
)

print("Reloaded model successfully")
print(f"model.device = {evalmodel.device}") #Verifying device

Reloaded model successfully
model.device = cuda:0


In [82]:
def answer(question, result_named):


    prompt = (
    f"Answer questions using the context. ALWAYS REFER TO THE EXAMPLES WHEN COMING UP WITH AN ANSWER."

    f"Example:\n"
    f"Question: What was the birthplace of Barack Obama/"
    f"Context: ['birthplace': Kapiolani Medical Center for Women and Children]"
    f"Answer: Barack Obama was born in the Kapiolani Medical Center for Women and Children"


    f"Question: {question}\n"
    f"Context: {context}\n"
    f"Answer:"
)


    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(evalmodel.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        temperature=0.1,
        eos_token_id=tokenizer.eos_token_id,
         no_repeat_ngram_size=3,

    )

    output_answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).split('\n')

    return output_answer

output_answer = answer(question, result_named)

print(output_answer)

['Lord of the Rings']
